In [35]:
import ase
import numpy as np
import pyscf
import time
import os
from pyscf.scf import hf
from pyscf import gto, dft, df, lib
from pyscf.gto import mole
import scipy

import torch
from equiv_dens.training.parse_command_line_arguments import parse_command_line_arguments
from equiv_dens.training.errors import ErrorDict
from equiv_dens.data.density_dataset import AtomsDensityData
from equiv_dens.data.hamiltonian_dataset import seeded_random_split
from equiv_dens.utils.grids import cubical_grid, cubical_sampling,\
     CubicalGrid, spherical_grid, spherical_radial_sampling
from equiv_dens.training.model_loader import load_model
import equiv_dens.utils.base as utils

from functools import partial

import ase.io
import dftd4.pyscf as d4disp

hf.MUTE_CHKFILE = True

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
# thio_poly = np.load('datasets/thiophene_all_test_pyscf_augccpvdz.npy', allow_pickle=true)
# save_file = 'datasets/thiophene_all_test_pyscf_d4_augccpvdz.npy'
thio_12mer = np.load('datasets/12mer_1-every20_pyscf_augccpvdz.npy', allow_pickle=True)
thio_12mer = list(thio_12mer)
print('thio 12mer len', len(thio_12mer))
save_file = 'datasets/12mer_1-every20_pyscf_d4_augccpvdz.npy'

thio 12mer len 21


In [37]:
if os.path.exists(save_file):
    results = list(np.load(save_file, allow_pickle=True))
else:
    results = []

print('results', results)
for i, calc_dict in enumerate(thio_12mer):
    print(i)
    start = time.time()
    mol = gto.Mole.unpack(calc_dict[0])
    mol.build()

    disp = d4disp.DFTD4Dispersion(mol, xc='pbe').kernel()
    print('calc time', time.time() - start)
    print('D4 energy', disp[0])
    print('D4 gradient', disp[1])
    print('calc energy', calc_dict[1]['energy'])
    print('calc forces', calc_dict[1]['forces'])
    if 'df_coeff' in calc_dict[1].keys():
        calc_dict[1].pop('df_coeff')

    calc_dict[1]['energy'] += disp[0]
    calc_dict[1]['forces'] += -disp[1]/ase.units.Bohr
    results.append(calc_dict)
    print((i+1) % 1000)
    if (i+1) % 1000 == 0:
        print('i=', i, 'saving file')
        # if (i+1) == 4000:
        #     break
        np.save(save_file, results, allow_pickle=True)

    # #print(mol.pack())
    # mf = dft.RKS(mol)
    # mf.chkfile=False
    # mf.xc = 'pbe'
    # #mf.max_cycle = 1000
    # mf.kernel()
    # g = mf.nuc_grad_method()
    # gradients = g.grad()
    # #print(mfs[i].mo_coeff)
    # res = []
    # res.append(mol.pack())
    # calc_dict = {}
    # print('mo occ', mf.mo_occ)
    # calc_dict['mo_coeff'] = mf.mo_coeff
    # calc_dict['mo_occ'] = mf.mo_occ
    # calc_dict['energy'] = mf.e_tot
    # calc_dict['forces'] = -gradients/ase.units.Bohr 

# pos = thio_3_mer['positions'][-3]
# atom_nums = thio_3_mer['atom_numbers'][-3]
#
# start = time.time()
# atom = []
# for j in range(len(atom_nums)):
#     atom.append((atom_nums[j], pos[j, :])) 
# basis = ''
# mol = gto.M(atom=atom, basis='augccpvdz')
# #print(mol.pack())
# mf = dft.RKS(mol)
# mf.chkfile=False
# mf.xc = 'pbe'
# mf.max_cycle = 1000
# mf.kernel()
# g = mf.nuc_grad_method()
# gradients = g.grad()
#
# print('dft energy', mf.e_tot)
# print('dft gradient', gradients)
#
# disp = d4disp.DFTD4Dispersion(mol, xc='pbe') 
# dkr = disp.kernel()
# print('D4 energy', dkr[0])
# print('D4 gradient', dkr[1])
np.save(save_file, results, allow_pickle=True)

results []
0
calc time 0.009649038314819336
D4 energy -0.16887747134586628
D4 gradient [[-5.88189210e-05  5.95337801e-05  4.00747441e-05]
 [-7.64795857e-05  4.26143106e-05 -8.36750321e-05]
 [-6.29510818e-05  8.25187284e-06 -1.55157492e-04]
 [-8.47714798e-05 -1.47690586e-04 -5.77635196e-05]
 [-1.06803987e-04 -3.57700820e-05 -2.47206060e-05]
 [ 5.22126752e-05 -1.04065662e-05  1.46115063e-04]
 [ 5.13740638e-06 -1.95694046e-05  1.65406537e-04]
 [-5.84989536e-06 -3.90943969e-05 -1.65114365e-04]
 [-3.50853772e-05 -3.65913052e-05 -1.90153171e-04]
 [ 5.33096716e-05  1.73865667e-04 -8.84113830e-05]
 [ 3.47917106e-05  1.34947223e-04 -1.06653578e-04]
 [-1.27695602e-06 -1.42116523e-04  4.06226452e-05]
 [-2.06509872e-05 -1.50274904e-04  5.19656777e-05]
 [-1.43321735e-05  4.53724756e-06  1.99173991e-04]
 [-1.15588513e-05  6.20150280e-06  1.66661179e-04]
 [ 5.54992165e-05 -4.78325500e-05 -1.39467731e-04]
 [-2.86217165e-05 -4.65735488e-05 -1.05598494e-04]
 [-5.16079856e-05  1.09404191e-04  1.38780068e

calc time 0.008093118667602539
D4 energy -0.16451229454242605
D4 gradient [[-6.28609946e-05  3.13236843e-05  6.72651107e-05]
 [-1.09951845e-04 -2.97453211e-05 -2.83857961e-05]
 [-1.08952313e-04 -8.32863924e-05 -8.85022855e-05]
 [ 3.58212411e-06 -9.83926515e-05 -1.21206415e-04]
 [-3.33082659e-05 -1.12788108e-04 -1.20209585e-04]
 [ 1.34539298e-05  1.46548603e-04  6.80296143e-05]
 [-3.10798479e-05  1.65337389e-04  5.68566696e-05]
 [ 9.85862121e-07 -6.83938325e-05 -1.55775098e-04]
 [-3.18889803e-05 -1.07379794e-04 -1.33374374e-04]
 [ 6.86847409e-05 -1.04562694e-05 -1.60337133e-04]
 [ 4.23972822e-05 -3.13837154e-05 -1.64562777e-04]
 [-4.86832229e-05  2.99133714e-06  1.52620763e-04]
 [-6.92824696e-05 -3.99858828e-05  1.47778937e-04]
 [-1.77928183e-05  9.86426156e-05  1.42590802e-04]
 [-4.53682248e-05  8.00106092e-05  1.38761217e-04]
 [ 2.80725191e-05 -2.84030781e-05 -1.66272003e-04]
 [-3.07170087e-06 -3.43767256e-05 -1.71533038e-04]
 [-2.35596913e-05  9.95327633e-05  1.19580323e-04]
 [-5.498

In [38]:
thiod4 = utils.calc_dict_to_npy(results, compress_atoms=False, convert_forces=False)
np.save('datasets/12mer_1-every20_pyscf_d4_augccpvdz_npy.npy', thiod4, allow_pickle=True)

In [1]:
print(i+1 %1000)

NameError: name 'i' is not defined

In [ ]:
mf = dft.RKS(mol)
mf.chkfile=False
mf.xc = 'pbe'
mf.max_cycle = 1000
d4mf = d4disp.energy(mf).run()
grad = d4mf.nuc_grad_method()
print('combined gradient', grad.kernel())

In [3]:
thio_no6 = np.load('datasets/thiophene_not_all_train_pyscf_d4_augccpvdz.npy', allow_pickle=True)
print(len(thio_no6))

4000


In [4]:
thio_6 = np.load('datasets/thiophene6mer_train_pyscf_d4_augccpvdz.npy', allow_pickle=True)
print(len(thio_6))

1000


In [5]:
print(thio_6.shape)
print(thio_no6.shape)

(1000, 2)
(4000, 2)


In [6]:
thio = np.concatenate([thio_no6, thio_6], axis=0)
np.save('datasets/thiophene_all_train_pyscf_d4_augccpvdz.npy', thio, allow_pickle=True)

In [2]:
thio_d4 = np.load('datasets/thiophene_all_test_pyscf_d4_augccpvdz.npy', allow_pickle=True)
thio_poly = np.load('datasets/thiophene_all_test_pyscf_augccpvdz.npy', allow_pickle=True)

In [4]:
indices = [214]
for i in indices:
    print(i)
    start = time.time()
    calc_dict = thio_d4[i]
    mol = gto.Mole.unpack(calc_dict[0])
    mol.build()

    mf = dft.RKS(mol)
    mf.chkfile = False
    mf.xc = 'pbe'
    mf.max_cycle = 1000
    d4mf = d4disp.energy(mf).run()
    grad = d4mf.nuc_grad_method()
    g = grad.kernel()
    print('d4mf.etot', d4mf.e_tot)
    print('calc en', calc_dict[1]['energy'])

    print('- grad to angstrom', -g/ase.units.Bohr)
    print('calc forces', calc_dict[1]['forces'])

214
converged SCF energy = -552.659181958268
--------------- DFTD4 gradients ---------------
         x                y                z
0 H    -0.0069831161     0.0004068013    -0.0077945764
1 S     0.0024302396     0.0043379074     0.0007689736
2 C     0.0334485500    -0.0259757192     0.0031550498
3 C    -0.0345995356     0.0120312217     0.0105186119
4 C     0.0225014166    -0.0085303411    -0.0053473800
5 C     0.0261518890     0.0059462941     0.0000525228
6 H     0.0018862242     0.0019591148     0.0021227917
7 H    -0.0109461547     0.0184174632    -0.0043461758
8 H    -0.0338843879    -0.0086150179     0.0008568328
----------------------------------------------
d4mf.etot -552.6591819582679
calc en -552.6591819582922
- grad to angstrom [[ 1.31961770e-02 -7.68743027e-04  1.47296147e-02]
 [-4.59248732e-03 -8.19745698e-03 -1.45314956e-03]
 [-6.32085988e-02  4.90869953e-02 -5.96217998e-03]
 [ 6.53836463e-02 -2.27357141e-02 -1.98772958e-02]
 [-4.25215148e-02  1.61200085e-02  1.0105

In [3]:
indices = [214, 1523, 2894]
for i in indices:
    print(i)
    start = time.time()
    calc_dict = thio_d4[i]
    mol = gto.Mole.unpack(calc_dict[0])
    mol.build()

    mf = dft.RKS(mol)
    mf.chkfile = False
    mf.xc = 'pbe'
    mf.max_cycle = 1000
    d4mf = d4disp.energy(mf).run()
    grad = d4mf.nuc_grad_method()
    g = grad.kernel()
    print('d4mf.etot', d4mf.e_tot)
    print('calc en', calc_dict[1]['energy'])

    print('- grad to angstrom', -g/ase.units.Bohr)
    print('calc forces', calc_dict[1]['forces'])

    mf.kernel()
    g = mf.nuc_grad_method()
    gradients = g.grad()
    #print(mfs[i].mo_coeff)
    print('mf etot', mf.e_tot)
    print('mf grads', -gradients/ase.units.Bohr)
    print('calc etot', thio_poly[i][1]['energy'])
    print('calc forces', thio_poly[i][1]['forces'])

    disp = d4disp.DFTD4Dispersion(mol, xc='pbe').kernel()
    print('disp en', disp[0])
    print('disp f', -disp[1]/ase.units.Bohr)

214
converged SCF energy = -552.656527066967
--------------- DFTD4 gradients ---------------
         x                y                z
0 H     0.0036875923    -0.0045903721    -0.0151561387
1 S     0.0142485311     0.0028474302    -0.0046771827
2 C     0.0030862264     0.0207280343     0.0401964805
3 C    -0.0692050121    -0.0316696982     0.0124277755
4 C     0.0269498855    -0.0098568763    -0.0401879103
5 C     0.0033516243    -0.0040191306    -0.0232718924
6 H     0.0302980790     0.0157490654    -0.0068325078
7 H     0.0129652573     0.0139664942     0.0159763451
8 H    -0.0253709880    -0.0031529686     0.0215259992
----------------------------------------------
d4mf.etot -552.6565270669671
calc en -552.6565270669709
- grad to angstrom [[-0.00696854  0.00867455  0.02864095]
 [-0.02692582 -0.00538086  0.00883859]
 [-0.00583212 -0.03917031 -0.07596034]
 [ 0.13077852  0.05984706 -0.02348509]
 [-0.0509279   0.0186268   0.07594414]
 [-0.00633365  0.00759506  0.0439775 ]
 [-0.057255

converged SCF energy = -1655.61181977125
--------------- DFTD4 gradients ---------------
         x                y                z
0 H    -0.0184253971     0.0015753698     0.0048318700
1 S     0.0081603956     0.0111587989     0.0027402060
2 C    -0.0005690226    -0.0153303679    -0.0036327844
3 C     0.0043853197    -0.0280655488     0.0055867333
4 C     0.0014528932    -0.0008446876    -0.0002715363
5 C     0.0120210537     0.0226851909    -0.0075196648
6 H     0.0190944687     0.0260516921    -0.0053786620
7 H    -0.0172970126     0.0055011708     0.0029768387
8 S    -0.0259910789    -0.0119783380     0.0016040371
9 C     0.0284484486    -0.0200935167    -0.0140920183
10 C    -0.0266887302    -0.0130406035     0.0246074983
11 C     0.0549829609     0.0447813117    -0.0218167911
12 C    -0.0005592921    -0.0343134606     0.0292200041
13 H     0.0062298537     0.0009387640    -0.0039635814
14 H    -0.0275570782    -0.0319074634     0.0002850883
15 S    -0.0186462000     0.01517245

In [3]:
thio_d4 = np.load('datasets/thiophene_all_train_pyscf_d4_augccpvdz.npy', allow_pickle=True)

thiod4 = utils.calc_dict_to_npy(thio_d4, compress_atoms=True)
np.save('datasets/thiophene_all_train_d4.npy', thiod4, allow_pickle=True)

In [4]:
for i in [100, 1100, 2100, 3100, 4100]:
    print('calc en', thio_d4[i][1]['energy'])
    print('npy en', thiod4['energy'][i])
    print('calc f', thio_d4[i][1]['forces'])
    print('npy f', thiod4['forces'][i])
    print('calc pos', thio_d4[i][0]['atom'])
    print('npy pos', thiod4['positions'][i])

calc en -552.658922121059
npy en [-552.65892212]
npy f [[-2.43143558e-02  2.95360922e-02  1.58274659e-02]
 [-3.69398136e-02 -1.35532874e-02 -6.51468384e-03]
 [-1.78621617e-02 -4.37539473e-02  9.80758189e-03]
 [-4.62093776e-02 -2.19798399e-02  1.18231698e-04]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 5.60518324e-02 -3.24713842e-03  2.27717071e-02]
 [ 1.86757951e-01 -6.87900150e-03 -4.31786825e-02]
 [-2.74785593e-02  1.76524506e-01  2.43558279e-02]
 [-1.86209659e-02 -1.36446993e-01 -1.07811064e-02]
 [ 0.00000000e+00  0.000000

NameError: name 'calc_dict' is not defined